In [35]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [36]:
df = pd.read_csv("/content/qoute_dataset.csv") #dataset of quotes

In [37]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [38]:
df.shape

(3038, 2)

In [39]:
quotes = df["quote"]
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


**Data Preprocessing**

(before applying lstm or anything as such , we nede to preprocess our data, and for that we needa use NLP techniques like vectorization , embeddings , tokenization etc)

In [40]:
quotes = quotes.str.lower()

In [41]:
#removing punctuations
import string
translator = str.maketrans('', '', string.punctuation) #for every punctuation replace it with empty space
quotes = quotes.apply(lambda x: x.translate(translator)) # "lambda x" -> every single quote in the dataset gets passed one by one



In [42]:
#tokenization (assign a number to each word)
from tensorflow.keras.preprocessing.text import Tokenizer

In [43]:
#the actual vocab size for our df can get pretty large sometimes(eg : 30lakh), but setting such a large number, we dont have enuf cpu nor gpu power for our model to compute that
vocab_size = 10000 #10k is a safe number which hopefully captures all the important words
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [44]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10] #8978 unique words and the top 10 most occuring words is printed

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [45]:
sequence = tokenizer.texts_to_sequences(quotes)

The quotes and their corresponding sequentially tokenized vectors are displayed

In [46]:
for i in range(3):
  print(quotes[i])


“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [47]:
for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


For our input and output variable, our project is sentence completion, i.e, we are actively predicting the next word

eg: look at the first quote, if we write "the" [713] , it should predict "world" [62]

similarly if we write "the world" [713,62]. output : "as" [29]

In [48]:
X = []
y = []

for seq in sequence:
  for i in range(1,len(seq)):
    X.append(seq[:i]) #input sequence
    y.append(seq[i]) #output sequence

In [49]:
max_len = 50

In [50]:
#padding
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X, maxlen=max_len, padding = "pre")

In [51]:
X_padded

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]], dtype=int32)

In [52]:
X_padded.shape

(85271, 50)

In [53]:
y = np.array(y)
y.shape

(85271,)

Our models like LSTM or GRU work with probabilities (0 and 1), so we need to one hot encode our output variable (y)
suppose y stores a tokenized word [121]
its one hot encoded to the vocab size, in it the position where the word is stored is encoded to 1 , the rest is 0, basically 9.9k 0's and only a single 1

[121] -> [0,0,0......1,.....0,0,0]

In [54]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SimpleRNN

In [55]:
embedding_units = 50
rnn_units = 128

**Note** : The code application of RNN LSTM and GRU is mostly similar, only the internal working is different

In [56]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_units, input_length=max_len) #this layer creates the embeddings
)

rnn_model.add(
    SimpleRNN(units=rnn_units) #the embeddings come under this simple RNN layer (128 units)
)
rnn_model.add(
    Dense(units=vocab_size, activation="softmax") #output layer
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [66]:
rnn_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [67]:
rnn_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 50, 50)         │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 128)            │        22,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10000)          │     1,290,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,812,912 (6.92 MB)

 Trainable params: 1,812,912 (6.92 MB)

 Non-trainable params: 0 (0.00 B)

In [59]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_units, input_length=max_len)
)
lstm_model.add(
    LSTM(units=rnn_units)
)
lstm_model.add(
    Dense(units=vocab_size, activation="softmax")
)

In [70]:
import tensorflow as tf
lstm_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy", tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name='top5')])
lstm_model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [68]:
epochs = 10
batch_size = 128
history_rnn = rnn_model.fit(X_padded, y, epochs=epochs, batch_size=batch_size, validation_split=0.1)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.1974 - loss: 4.4653 - val_accuracy: 0.1059 - val_loss: 6.5476
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.2163 - loss: 4.1898 - val_accuracy: 0.1040 - val_loss: 6.6489
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.2412 - loss: 3.9708 - val_accuracy: 0.1037 - val_loss: 6.7636
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.2652 - loss: 3.7798 - val_accuracy: 0.1019 - val_loss: 6.8632
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.2916 - loss: 3.6139 - val_accuracy: 0.0971 - val_loss: 6.9772
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.3136 - loss: 3.4673 - val_accuracy: 0.0956 - val_loss: 7.0800
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.3342 - loss: 3.3381 - val_accuracy: 0.0944 - val_loss: 7.1660
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.3529 - loss: 3.2215 - val_accuracy: 

In [72]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

history_lstm = lstm_model.fit(
    X_padded, y,
    epochs=100,
    batch_size=128,
    validation_split=0.1,

)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.1143 - loss: 5.5149 - top5: 0.2640 - val_accuracy: 0.1035 - val_loss: 6.4820 - val_top5: 0.2397
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.1239 - loss: 5.3515 - top5: 0.2777 - val_accuracy: 0.1086 - val_loss: 6.5173 - val_top5: 0.2438
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.1309 - loss: 5.1981 - top5: 0.2896 - val_accuracy: 0.1080 - val_loss: 6.5589 - val_top5: 0.2489
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.1381 - loss: 5.0501 - top5: 0.3028 - val_accuracy: 0.1115 - val_loss: 6.6056 - val_top5: 0.2502
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.1457 - loss: 4.9098 - top5: 0.3125 - val_accuracy: 0.1115 - val_loss: 6.6595 - val_top5: 0.2528
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.1531 - loss: 4.7762 - top5: 0.3250 - val_accuracy: 0.1121 - val_loss: 6.7263 - val_top5: 0.2528
Epoch 7/100
6

In [73]:
lstm_model.save("lstm_model.h5")

In [74]:
#word map
index_to_word = {}
for word,index in word_index.items():
  index_to_word[index] = word

In [80]:
def predictor(text,model,tokenizer, max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre') #sequentially tokenize the recieved text and adds padding

  preds = model.predict(seq, verbose=0)
  pred_index = np.argmax(preds)
  pred_word = index_to_word[pred_index]

  return pred_word

In [84]:
testing_text = "what are you "
predicted_word = predictor(testing_text, lstm_model, tokenizer, max_len)
print(predicted_word)

implying


In [85]:
def generate_text(seed_text, n_words, model, tokenizer, max_len):
    for _ in range(n_words):
        next_word = predictor(seed_text, model, tokenizer, max_len)
        if next_word == "":
          break
        seed_text += " " + next_word

    return seed_text

In [87]:
text_to_complete = "a thousand "
print(generate_text(text_to_complete, len(text_to_complete), lstm_model, tokenizer, max_len))

a thousand  times without laughter is a question to the heavens heart it


In [88]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokenizer, f)

In [89]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)